In [1]:
# Engagement Model
#
# Question:
# How does test engagement behavior relate to signup probability,
# controlling for device and geography?
#
# Population:
# All visitors
#
# Outcome:
# has_signup (binary)

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

import statsmodels.api as sm

In [4]:
DATA_PATH = "../data/visitor_features_engineered.parquet"

df = pd.read_parquet(DATA_PATH)

df.shape

(535391, 15)

In [5]:
TARGET = "has_signup"

CATEGORICAL_FEATURES = [
    "browser",
    "os_name_clean",
    "sub_region",
    "test_engagement_state",
]

df_model = df[CATEGORICAL_FEATURES + [TARGET]].copy()
df_model[TARGET] = df_model[TARGET].astype(int)

df_model.head()

,browser,os_name_clean,sub_region,test_engagement_state,has_signup
0,mobile_web,iOS,East South Central,No Test,0
1,mobile_web,iOS,East South Central,No Test,0
2,mobile_web,iOS,East South Central,Test Started Only,0
3,mobile_web,iOS,East South Central,Test Started Only,0
4,mobile_web,Android,East South Central,Test Started Only,0


In [6]:
for col in CATEGORICAL_FEATURES:
    print(f"\n{col}")
    print(df_model[col].value_counts())


browser
browser
mobile_web     453122
desktop_web     82269
Name: count, dtype: int64

os_name_clean
os_name_clean
iOS          329050
Android      147286
Windows       31192
macOS         14163
Other          7039
Chrome OS      6661
Name: count, dtype: int64

sub_region
sub_region
South Atlantic        111085
East North Central     82372
Pacific                74678
West South Central     66942
Middle Atlantic        63095
Mountain               43274
East South Central     37575
West North Central     35547
New England            20823
Name: count, dtype: int64

test_engagement_state
test_engagement_state
Test Completed       425185
Test Started Only     77550
No Test               32656
Name: count, dtype: int64


In [7]:
REFERENCE_CATEGORIES = {
    "browser": "mobile_web",
    "os_name_clean": "iOS",
    "sub_region": "South Atlantic",
    "test_engagement_state": "No Test",
}

In [8]:
encoder = OneHotEncoder(
    drop=None,
    sparse_output=False,
    handle_unknown="ignore"
)

encoder.set_output(transform="pandas")

X_encoded = encoder.fit_transform(df_model[CATEGORICAL_FEATURES])

In [9]:
X_encoded.head()

,browser_desktop_web,browser_mobile_web,os_name_clean_Android,os_name_clean_Chrome OS,os_name_clean_Other,os_name_clean_Windows,os_name_clean_iOS,os_name_clean_macOS,sub_region_East North Central,sub_region_East South Central,sub_region_Middle Atlantic,sub_region_Mountain,sub_region_New England,sub_region_Pacific,sub_region_South Atlantic,sub_region_West North Central,sub_region_West South Central,test_engagement_state_No Test,test_engagement_state_Test Completed,test_engagement_state_Test Started Only
0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [10]:
for col, ref in REFERENCE_CATEGORIES.items():
    ref_col = f"{col}_{ref}"
    if ref_col in X_encoded.columns:
        X_encoded = X_encoded.drop(columns=ref_col)

In [11]:
X_encoded.head()

,browser_desktop_web,os_name_clean_Android,os_name_clean_Chrome OS,os_name_clean_Other,os_name_clean_Windows,os_name_clean_macOS,sub_region_East North Central,sub_region_East South Central,sub_region_Middle Atlantic,sub_region_Mountain,sub_region_New England,sub_region_Pacific,sub_region_West North Central,sub_region_West South Central,test_engagement_state_Test Completed,test_engagement_state_Test Started Only
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [12]:
X = sm.add_constant(X_encoded)
y = df_model[TARGET]

X.shape, y.mean()

((535391, 17), np.float64(0.01630023664947674))

In [13]:
logit = sm.Logit(y, X)
result = logit.fit(disp=False)

result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:             has_signup   No. Observations:               535391
Model:                          Logit   Df Residuals:                   535374
Method:                           MLE   Df Model:                           16
Date:                Fri, 16 Jan 2026   Pseudo R-squ.:                 0.04672
Time:                        12:27:28   Log-Likelihood:                -42498.
converged:                       True   LL-Null:                       -44581.
Covariance Type:            nonrobust   LLR p-value:                     0.000
===========================================================================================================
                                              coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------
const                                      -6.5315      0.146    -44.674      0.000      -6.818      -6.245
browser_desktop_web                        -1.4418      0.097    -14.828      0.000      -1.632      -1.251
os_name_clean_Android                       0.1983      0.024      8.404      0.000       0.152       0.245
os_name_clean_Chrome OS                     1.6260      0.129     12.580      0.000       1.373       1.879
os_name_clean_Other                         0.0550      0.196      0.281      0.779      -0.329       0.439
os_name_clean_Windows                       0.9928      0.111      8.964      0.000       0.776       1.210
os_name_clean_macOS                         0.9446      0.128      7.361      0.000       0.693       1.196
sub_region_East North Central              -0.0385      0.036     -1.063      0.288      -0.110       0.033
sub_region_East South Central               0.2285      0.043      5.337      0.000       0.145       0.312
sub_region_Middle Atlantic                 -0.0696      0.040     -1.737      0.082      -0.148       0.009
sub_region_Mountain                        -0.1960      0.047     -4.139      0.000      -0.289      -0.103
sub_region_New England                     -0.2302      0.066     -3.501      0.000      -0.359      -0.101
sub_region_Pacific                         -0.1849      0.039     -4.698      0.000      -0.262      -0.108
sub_region_West North Central              -0.0369      0.048     -0.770      0.441      -0.131       0.057
sub_region_West South Central               0.1131      0.037      3.064      0.002       0.041       0.185
test_engagement_state_Test Completed        2.7059      0.145     18.677      0.000       2.422       2.990
test_engagement_state_Test Started Only    -1.0342      0.216     -4.795      0.000      -1.457      -0.611
===========================================================================================================
"""

In [14]:
odds_ratios = pd.DataFrame({
    "odds_ratio": np.exp(result.params),
    "ci_lower": np.exp(result.conf_int()[0]),
    "ci_upper": np.exp(result.conf_int()[1]),
    "p_value": result.pvalues
})

odds_ratios.sort_values("odds_ratio", ascending=False)

,odds_ratio,ci_lower,ci_upper,p_value
test_engagement_state_Test Completed,14.968104,11.268011,19.883201,7.565248e-78
os_name_clean_Chrome OS,5.083344,3.945755,6.548909,2.725412e-36
os_name_clean_Windows,2.698770,2.172171,3.353031,3.125987e-19
os_name_clean_macOS,2.571751,1.999888,3.307137,1.820149e-13
sub_region_East South Central,1.256767,1.155590,1.366803,9.454498e-08
os_name_clean_Android,1.219355,1.164243,1.277075,4.302212e-17
sub_region_West South Central,1.119710,1.041595,1.203683,2.180512e-03
os_name_clean_Other,1.056515,0.719658,1.551047,7.789933e-01
sub_region_West North Central,0.963778,0.877433,1.058619,4.410486e-01
sub_region_East North Central,0.962211,0.896235,1.033044,2.878227e-01


In [15]:
# Interpretation reminders:
#
# - Odds Ratio > 1: higher odds of signup vs reference
# - Odds Ratio < 1: lower odds of signup vs reference
# - Reference visitor:
#   - mobile_web
#   - iOS
#   - South Atlantic
#   - No Test